### Analysis of voltage dynamics during passive DoC

#### Notebook goals
- Validate extracted voltage data for each session, DMD, and ROI -- ensure quality alignment to image presentations
- Recapitulate analyses of image presentation dynamics (response variance, sequence dynamics) for voltage as in glutamate
- Summarize voltage dynamics per session, create a comprehensive idea of voltage dynamics correlations between ROIs on the same DMD and between DMDs
#### Specific analyses
- Image selectivity analyses x session x depth (variance, FVE, % dFF)
- Image sequence analysis
- Image change and omission analyses -- differences between depths, sessions, FVE by change relative to mean image response
- Voltage oscillation analysis
#### Order of operations
1) Gather relevant sessions -- sort, organize, and open
2) Plot example image PSTH for each ROI, DMD, session
3) Perform/classify dendrites as either image activated or not 


In [ ]:
import sys
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry

display(HTML("<style>.container { width:100% !important; }</style>"))
warnings.filterwarnings("default")

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [ ]:
# ---------------------------------------------------------------------
# Local repository / data paths
# ---------------------------------------------------------------------
# If running from inside the repo, this can usually stay as None.
# If imports fail, set REPO_ROOT to your local clone, e.g.
# REPO_ROOT = Path(r"C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis")
REPO_ROOT = None

if REPO_ROOT is not None:
    src_path = Path(REPO_ROOT) / "src"
    if str(src_path) not in sys.path:
        sys.path.insert(0, str(src_path))

BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")

# Optional local copy of the summary table. The registry discovers sessions from BASE_PATH;
# this file is only useful for ad hoc inspection or manual cross-checks.
SUMMARY_XLSX = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\VIP_SD_summary.xlsx")

# ---------------------------------------------------------------------
# Session selection
# ---------------------------------------------------------------------
target_mice = [
852835, 863774
]

EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]
PARADIGMS = ["change_detection_passive"]

In [ ]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=target_mice,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions for target mice: {target_mice}")
display(process_df.head())

In [ ]:
process_df

In [ ]:
idx = 4
test_path = assets[idx].derived_dir / 'voltage' / 'voltage_mean_dff_robust_f0_trial.npz'
print(f'Session: {assets[idx].session_id}')

In [ ]:
data = np.load(test_path,allow_pickle=True)['data'][0]

In [ ]:
im_names = list(data['DMD1']['image_identity'].keys())
colors = ['#c5cae9', '#ffcdd2', '#c8e6c9', '#ffe0b2',
 '#e1bee7', '#d7ccc8', '#cfd8dc', '#b2ebf2']

In [ ]:
dmd = 1
freq = data['metadata']['sample_rate_hz']
dmd_data = data[f'DMD{dmd}']
n_rois = len(dmd_data['roi_ids'])

### Plot change events

In [ ]:
window = data['metadata']['windows_sec']['change']
n_events = data[f'DMD{dmd}']['change']['n_events']

for dend in range(n_rois):
    fig,ax=plt.subplots()
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
    ax.set_xlabel('Time (s) from image change')
    ax.set_ylabel('\u0394F/F$_{0}$')
    
    change_mean = dmd_data['change']['mean'][dend]
    change_std = dmd_data['change']['std'][dend]/np.sqrt(n_events)
    t = np.linspace(-window[0],window[1],len(change_mean))
    ax.plot(t,change_mean,color='k')
    
    ax.fill_between(t,change_mean-change_std,change_mean+change_std,alpha=0.4,zorder=0,color='lightgray')
    ax.axvspan(-0.75,-0.5,alpha=0.2,zorder=0,color='gray')
    ax.axvspan(0.0,0.25,alpha=0.2,zorder=0)
    
    ax.set_title(f'ROI {dend}')
    
fig.tight_layout()

### Plot omission events

In [ ]:
window = data['metadata']['windows_sec']['omission']
n_events = data[f'DMD{dmd}']['omission']['n_events']

for dend in range(n_rois):
    fig,ax=plt.subplots()
    
    ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
    ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)
    
    omission_mean = dmd_data['omission']['mean'][dend]
    omission_std = dmd_data['omission']['std'][dend]/np.sqrt(n_events)
    t = np.linspace(-window[0],window[1],len(omission_mean))
    ax.plot(t,omission_mean,color='k')
    
    ax.fill_between(t,omission_mean-omission_std,omission_mean+omission_std,alpha=0.4,zorder=0,color='lightgray')
    ax.axvspan(-0.75,-0.5,alpha=0.2,zorder=0)
    ax.axvline(0.0,color='lightgray',dashes=[3,3],zorder=0)
    ax.axvline(0.25,color='lightgray',dashes=[3,3],zorder=0)
    ax.axvspan(0.75,1.0,alpha=0.2,zorder=0)
    ax.set_title(f'ROI {dend}')
    ax.set_xlabel('Time (s) from image omission')
    ax.set_ylabel('\u0394F/F$_{0}$')
fig.tight_layout()

### Image presentations

In [ ]:
window = data['metadata']['windows_sec']['image']

for dend in range(n_rois):
    fig,ax=plt.subplots()
    mean = []
    
    ax.axvspan(0.0,0.25,color='lightgray',alpha=0.2)
    
    for i,im in enumerate(dmd_data['image_identity'].keys()):
        im_data = dmd_data['image_identity'][im]['mean'][dend]
        t = np.linspace(-window[0],window[1],len(im_data))
        ax.plot(t,im_data-np.mean(im_data[:int(freq/4)]),color=colors[i])
        mean.append(im_data)
    mean = np.mean(mean,axis=0)
    ax.plot(t,mean-np.mean(mean[:int(freq/4)]),color='k')
        
fig.tight_layout()